# Domain- und URL-Uebersicht fuer die Praesentation

Dieses Notebook zeigt:
- Welche Domains genannt wurden
- Wie viele verschiedene URLs pro Domain vorkommen
- Detailansicht fuer `statistik-berlin-brandenburg.de` mit verschiedenen URL-Pfaden

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

pd.set_option('display.max_colwidth', None)

# Nutzt die aggregierten URL-Daten mit Domain-Spalte
base = Path.cwd()
csv_path = (base / '..' / 'data' / 'analysis' / 'top_urls.csv').resolve()
if not csv_path.exists():
    # Fallback, falls das Notebook aus dem Projekt-Root gestartet wird
    csv_path = (base / 'data' / 'analysis' / 'top_urls.csv').resolve()

df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Tabelle: gleiche Domain, wie viele verschiedene URLs
domain_table = (
    df.groupby('domain', as_index=False)
      .agg(
          verschiedene_urls=('url', 'nunique'),
          summe_occurrences=('occurrences', 'sum'),
          summe_runs=('runs', 'sum')
      )
      .sort_values(['verschiedene_urls', 'summe_occurrences'], ascending=[False, False])
      .reset_index(drop=True)
)

# Nur Domains mit mehr als einer URL fuer den Praesentationsfokus
domain_mehrere_urls = domain_table[domain_table['verschiedene_urls'] > 1].copy()
domain_mehrere_urls

In [ ]:
# Detail: statistik-berlin-brandenburg.de mit verschiedenen Pfaden
target_domain = 'statistik-berlin-brandenburg.de'
focus = df[df['domain'].eq(target_domain)].copy()

focus['pfad'] = focus['url'].apply(lambda u: urlparse(u).path or '/')
focus = focus.sort_values(['occurrences', 'url'], ascending=[False, True]).reset_index(drop=True)

print(f"Anzahl unterschiedlicher URLs fuer {target_domain}: {focus['url'].nunique()}")
focus[['domain', 'pfad', 'url', 'occurrences', 'runs']]

In [ ]:
# Optional: Tabellen fuer die Folien als CSV exportieren
out_dir = (Path.cwd() / 'analysis_outputs').resolve()
if not out_dir.exists():
    out_dir = (Path.cwd() / 'notebooks' / 'analysis_outputs').resolve()
out_dir.mkdir(parents=True, exist_ok=True)

domain_mehrere_urls.to_csv(out_dir / 'domain_mehrere_urls.csv', index=False)
focus[['domain', 'pfad', 'url', 'occurrences', 'runs']].to_csv(out_dir / 'statistik_berlin_brandenburg_urls.csv', index=False)

print('Exportiert nach:', out_dir)

## Weitere Folienbausteine

- URL vs. Domain: Domains sind stabiler als URLs
- Stabile Kerndomains: Einige Anbieter tauchen immer wieder auf
- Kategorienvergleich: Prozedurale Fragen waren am stabilsten

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import FancyBboxPatch

pd.set_option('display.max_colwidth', None)

current_dir = Path.cwd()
root_dir = current_dir if (current_dir / 'data' / 'analysis' / 'stability_by_prompt.csv').exists() else current_dir.parent
if not (root_dir / 'data' / 'analysis' / 'stability_by_prompt.csv').exists():
    root_dir = root_dir.parent

analysis_dir = root_dir / 'data' / 'analysis'
out_dir = root_dir / 'notebooks' / 'analysis_outputs'
out_dir.mkdir(parents=True, exist_ok=True)

stability = pd.read_csv(analysis_dir / 'stability_by_prompt.csv')
category = pd.read_csv(analysis_dir / 'category_summary.csv')
top_domains = pd.read_csv(analysis_dir / 'top_domains.csv')

focus_prompts = ['P03', 'P06', 'P07']
focus = stability[stability['prompt_id'].isin(focus_prompts)].copy().sort_values('prompt_id')

fig, ax = plt.subplots(figsize=(8.8, 4.8))
x = range(len(focus))
bar_width = 0.36
ax.bar([i - bar_width / 2 for i in x], focus['mean_url_jaccard'], width=bar_width, label='URL-Jaccard', color='#d95f02')
ax.bar([i + bar_width / 2 for i in x], focus['mean_domain_jaccard'], width=bar_width, label='Domain-Jaccard', color='#1b9e77')
ax.set_xticks(list(x))
ax.set_xticklabels(focus['prompt_id'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Jaccard-Aehnlichkeit')
ax.set_title('URL vs. Domain: Domains sind stabiler als URLs')
ax.legend(frameon=False, ncols=2, loc='upper center', bbox_to_anchor=(0.5, 1.12))
ax.grid(axis='y', alpha=0.2)

for idx, row in focus.reset_index(drop=True).iterrows():
    ax.text(idx - bar_width / 2, row['mean_url_jaccard'] + 0.025, f"{row['mean_url_jaccard']:.2f}", ha='center', va='bottom', fontsize=9)
    ax.text(idx + bar_width / 2, row['mean_domain_jaccard'] + 0.025, f"{row['mean_domain_jaccard']:.2f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(out_dir / 'url_vs_domain_p03_p06_p07.png', dpi=200, bbox_inches='tight')
plt.show()

cards = top_domains.head(6).copy()
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
fig.suptitle('Stabile Kerndomains: Einige Anbieter tauchen immer wieder auf', fontsize=16, fontweight='bold')
axes_flat = axes.flatten()
for ax_card, row in zip(axes_flat, cards.itertuples(index=False)):
    ax_card.set_axis_off()
    card = FancyBboxPatch((0.03, 0.08), 0.94, 0.84, boxstyle='round,pad=0.02,rounding_size=0.04', linewidth=1.5, edgecolor='#2f2f2f', facecolor='#f7f4ef')
    ax_card.add_patch(card)
    ax_card.text(0.5, 0.72, row.domain, ha='center', va='center', fontsize=12, fontweight='bold')
    ax_card.text(0.5, 0.48, f"{int(row.occurrences)} Nennungen", ha='center', va='center', fontsize=15, color='#1b9e77')
    ax_card.text(0.5, 0.26, f"{int(row.prompts)} Prompts | {int(row.runs)} Runs", ha='center', va='center', fontsize=10, color='#555555')
for ax_card in axes_flat[len(cards):]:
    ax_card.set_axis_off()
plt.tight_layout()
plt.savefig(out_dir / 'stable_core_domains_cards.png', dpi=200, bbox_inches='tight')
plt.show()

cat = category.copy().sort_values('mean_domain_jaccard', ascending=False)
fig, ax = plt.subplots(figsize=(8.8, 4.8))
cat_x = range(len(cat))
ax.bar([i - bar_width / 2 for i in cat_x], cat['mean_url_jaccard'], width=bar_width, label='URL-Jaccard', color='#d95f02')
ax.bar([i + bar_width / 2 for i in cat_x], cat['mean_domain_jaccard'], width=bar_width, label='Domain-Jaccard', color='#1b9e77')
ax.set_xticks(list(cat_x))
ax.set_xticklabels(cat['category'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Jaccard-Aehnlichkeit')
ax.set_title('Kategorienvergleich: Prozedurale Fragen waren am stabilsten')
ax.legend(frameon=False, ncols=2, loc='upper center', bbox_to_anchor=(0.5, 1.12))
ax.grid(axis='y', alpha=0.2)

for idx, row in cat.reset_index(drop=True).iterrows():
    ax.text(idx - bar_width / 2, row['mean_url_jaccard'] + 0.025, f"{row['mean_url_jaccard']:.2f}", ha='center', va='bottom', fontsize=9)
    ax.text(idx + bar_width / 2, row['mean_domain_jaccard'] + 0.025, f"{row['mean_domain_jaccard']:.2f}", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(out_dir / 'category_comparison_stability.png', dpi=200, bbox_inches='tight')
plt.show()

summary = pd.DataFrame([
    {
        'Aussage': 'Domains sind stabiler als URLs',
        'Beleg': 'P03 zeigt sehr niedrige URL-Aehnlichkeit, aber hohe Domain-Aehnlichkeit.'
    },
    {
        'Aussage': 'Stabile Kerndomains',
        'Beleg': 'Mehrere Domains tauchen in vielen Antworten wieder auf und sind gute Anker fuer die Folie.'
    },
    {
        'Aussage': 'Prozedurale Fragen stabiler',
        'Beleg': 'Die Kategorie Prozedurale Frage hat die hoechste mittlere Domain-Aehnlichkeit.'
    }
])
summary

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\lenaf\\vscProjects\\llm-quellenkonsistenz\\notebooks\\data\\analysis\\stability_by_prompt.csv'